Protocolo vigente: una trayectoria de 100 generaciones de carácter evolutivo (algoritmo evolutivo poblacional), sin diseño base. Luz 20 m, alturas y secciones variables. Objetivos: masa mínima, utilización promedio máxima y desplazamiento Z mínimo. Criterio principal L/700; transferencia cronológica complementaria. Los resultados previos deben regenerarse tras el entrenamiento.


# 01 — Datos y metodología

**Pregunta principal:** ¿Cómo se comparan distintos modelos de aprendizaje automático en desempeño predictivo, estabilidad y costo computacional al estimar masa, utilización promedio y desplazamiento de cerchas de acero en 2D, y qué utilidad ofrecen sus predicciones para seleccionar diseños de mayor calidad multiobjetivo bajo presupuestos limitados de consulta?

La comparación predictiva es el objetivo principal. El segundo modelo es una política de búsqueda de diseños registrados: recupera alternativas Pareto de menor masa, menor desplazamiento y mayor utilización promedio bajo restricciones agregadas. No es un modelo inverso supervisado ni certifica un óptimo global.

`make prepare` comprueba respuestas de duplicados y conserva diseños únicos. Este notebook lee los datos preparados. Confirmar con la profesora el procedimiento y las condiciones físicas de origen sigue siendo necesario.


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from truss_research.evaluacion.presentacion import notebook_context

context = notebook_context()
STUDY_DIR, REPORT_DIR, CONFIG = context.study_dir, context.report_dir, context.config
DATA_DIR, STATE, RECORDS = CONFIG.data_dir, context.state, context.records
display(
    Markdown(
        f"**Estudio:** `{STUDY_DIR.name}` · **etiqueta:** `{CONFIG.label}` · **informe:** `{REPORT_DIR.name}`"
    )
)
if CONFIG.label == "comprobacion":
    display(
        Markdown(
            "**Comprobación técnica breve: sus métricas no constituyen resultados de tesis.**"
        )
    )

## 1. Reconocer entradas, respuestas y unidades

`web_config` identifica categorías, no una escala numérica física. Los identificadores permiten rastrear el origen y no se entregan como entradas al predictor. La masa está en kg, el desplazamiento en m y la utilización es un ratio.


In [ ]:
from truss_research.cerchas.schemas import INPUT_COLUMNS, OUTPUT_COLUMNS
from truss_research.datos.procesados import load_prepared_split, read_manifest

manifest = read_manifest(DATA_DIR)
split = load_prepared_split(DATA_DIR)
training = split.training
roles = pd.DataFrame(
    {
        "variable": [*INPUT_COLUMNS, *OUTPUT_COLUMNS],
        "rol": ["entrada"] * len(INPUT_COLUMNS) + ["respuesta"] * len(OUTPUT_COLUMNS),
    }
)
display(roles)
display(training.loc[:, ["iter_id", *INPUT_COLUMNS, *OUTPUT_COLUMNS]].head())

## 2. Detectar repeticiones y comprender la dinámica evolutiva

Los 5.000 registros corresponden a 100 generaciones de carácter evolutivo (0 a 99), generadas por un algoritmo evolutivo poblacional en lotes de 50 consultas de evaluación (0 a 49). Cada ciclo generacional evalúa 50 propuestas que buscan optimizar los diseños previos mediante operadores evolutivos (cruce y mutación), pudiendo resultar mejores, equivalentes o peores.

Dos registros con las mismas siete entradas representan el mismo diseño exacto. Su alta repetición (3.727 duplicados frente a 1.273 únicos) se debe al elitismo del algoritmo y a la convergencia de la población. La preparación comprueba que sus respuestas coincidan y conserva un representante único junto al registro completo de pertenencias, evitando que un modelo sea evaluado con una copia exacta vista en entrenamiento.


In [ ]:
display(pd.Series(manifest["design_audit"], name="Auditoría del origen"))
membership = pd.read_csv(DATA_DIR / "design_membership.csv")
display(membership[["iter_id", "representative_iter_id", "split"]].head(8))

## 3. Comprobar la separación y la cobertura

La partición principal usa diseños únicos, semilla 42 y estratificación por categoría. Compartir un identificador de generación entre conjuntos es posible; compartir un diseño exacto no lo es.

El análisis cronológico complementario fija bloques de generaciones y purga todo diseño compartido. Mide transferencia dentro de una sola trayectoria; no representa nuevos problemas estructurales independientes.


In [ ]:
from truss_research.datos.auditoria import audit_split_overlap

counts = (
    pd.DataFrame(
        {
            name: table["web_config"].value_counts().sort_index()
            for name, table in split.as_mapping().items()
        }
    )
    .fillna(0)
    .astype(int)
)
display(counts)
display(pd.Series(audit_split_overlap(split), name="Diseños exactos compartidos"))
counts.plot.bar(
    figsize=(9, 4), ylabel="Diseños", title="Cobertura por categoría y conjunto"
)
plt.tight_layout()
plt.show()

## 4. Explorar únicamente entrenamiento

Las decisiones de modelado se apoyan en entrenamiento. Las categorías poco representadas y los diseños muy parecidos limitan las conclusiones, aunque no existan duplicados exactos. Las correlaciones describen asociaciones; no prueban causas físicas.

La prueba ya fue observada: una campaña nueva sigue siendo un análisis posterior. Los límites actuales son operativos y no certifican seguridad por barra.


In [ ]:
import seaborn as sns

figure, axes = plt.subplots(1, len(OUTPUT_COLUMNS), figsize=(12, 3.5))
for axis, target in zip(axes, OUTPUT_COLUMNS, strict=True):
    sns.histplot(training[target], ax=axis)
    axis.set_title(target)
plt.tight_layout()
plt.show()
numeric = [name for name in INPUT_COLUMNS if name != "web_config"]
figure, axis = plt.subplots(figsize=(9, 7))
sns.heatmap(
    training[[*numeric, *OUTPUT_COLUMNS]].corr(),
    cmap="coolwarm",
    center=0,
    annot=True,
    fmt=".2f",
    ax=axis,
)
plt.tight_layout()
plt.show()

## Conclusión de esta etapa

El tamaño relevante para modelado es la cantidad de diseños únicos (1.273). Conservar los incumplidores permite estudiar errores de cumplimiento. La utilización promedio es una variable continua en [0, inf) cuyo límite operativo admisible es paramétrico (por defecto 1,0, pero ampliable si se evalúan tolerancias); junto con el desplazamiento admisible (≤ 20/700 m), constituyen umbrales de servicio del experimento que no sustituyen una certificación de seguridad barra por barra.


## Diagnósticos de robustez
Los errores extremos se conservan y se presentan por separado. La cobertura escasa y la procedencia física pendiente limitan las conclusiones; no se retocan modelos a partir de prueba.

In [ ]:
from truss_research.evaluacion.robustez import EVIDENCE_NOTES

display(Markdown("\n\n".join(EVIDENCE_NOTES.values())))
for diagnostic in (
    "error_tail_summary.csv",
    "category_coverage.csv",
    "physical_provenance.csv",
    "predictive_seed_variation.csv",
    "convergence_diagnostics.csv",
):
    diagnostic_path = REPORT_DIR / diagnostic
    if diagnostic_path.is_file():
        display(pd.read_csv(diagnostic_path).head(20))

## 5. Evidencia disponible

Las categorías 1 y 2 concentran 1.204 de 1.273 diseños. En la muestra registrada la utilización promedio máxima observada es 0,913 (no hay casos > 1 por el filtrado del optimizador original) y solo 20 casos exceden 20/700 m. Son frecuencias del banco ya conocido; no se usan para ajustar umbrales después de mirar prueba. La falta de ejemplos observados con utilización > 1 limita la evaluación empírica de fallos severos de resistencia.


## Contrato científico y procedencia

El título oficial se mantiene. El objetivo compara desempeño predictivo, estabilidad del ajuste final y costo, y evalúa utilidad para seleccionar diseños del catálogo. La pregunta y su matriz de evidencia están en CONTEXT.md. La estabilidad del desempeño no es estabilidad cinemática. La ficha docs/SCOPE.md distingue información confirmada, informada por el autor y pendiente. El origen físico sigue provisional.


## Reserva independiente y pertenencias

El ajuste y la calibración tienen funciones separadas. La sensibilidad por generaciones purga todos los diseños que cruzan particiones; los tamaños efectivos y categorías ausentes delimitan su interpretación.


In [ ]:
for filename in [
    "chronological_category_coverage.csv",
    "chronological_purged_designs.csv",
]:
    path = REPORT_DIR / filename
    if path.is_file():
        display(Markdown("**" + filename + "**"))
        display(pd.read_csv(path).head(30))